# 인공지능 보안 3주차 실습 — 학생용

Raw Log → Parsing → DataFrame/CSV → 전처리 → 정규화 → 통합 → 탐색·시각화

**합성 교육 데이터**입니다. 실제 공격이나 실사용자 기록이 아니며 문서용 IP 대역과 사설 IP를 사용합니다. 총 35개 실습, 코드 문제 8개와 서술 문제 1개입니다. 학생용의 `None`은 답을 넣을 자리이며 다음 단계 실행을 막지 않습니다. 정답은 해당 변수에 직접 작성하세요.

**50분씩 3차시**: 1차시 PPT와 실습 1~4, 2차시 실습 5~21, 3차시 22~35. 제공된 긴 파서는 실행·읽기 중심으로 진행합니다. 문제 7~9는 마지막 종합 활동이며 시간이 부족하면 복습 과제로 이어갑니다.

## 실습 파일 업로드
PC의 네 원본 파일 `web_access.log`, `auth.log`, `firewall.log`, `network_flow.log`를 선택하세요. 여러 파일을 한 번에 선택할 수 있습니다. 아래 셀을 실행하면 Colab의 파일 선택 창이 나타납니다. CSV는 지금 업로드할 필요가 없습니다. 실습 10에서 `teaching_metadata.csv`만 추가 업로드합니다.

업로드 파일은 현재 실행 세션에 저장됩니다. 런타임이 초기화되거나 종료되면 사라질 수 있으므로 다시 업로드하세요. Google Drive 연결이나 별도 패키지 설치가 필요하지 않습니다.

로컬 Jupyter에서 실행할 때는 같은 폴더에 파일을 두면 됩니다. 아래 코드는 Colab에서는 반드시 업로드 창을 열고, 로컬에서는 파일 존재만 확인합니다.

In [ ]:
# IN_COLAB: 현재 실행 환경이 Google Colab이면 True, 일반 Python 환경이면 False입니다.
# uploaded: {파일명: 파일 내용(bytes)} 형태의 사전입니다. 업로드한 파일은 현재 작업 폴더에도 저장됩니다.
import os
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    print(uploaded.keys())
else:
    print('로컬 실행: 현재 폴더의 배포 파일을 사용합니다.')
# required: 파싱을 시작하기 전에 필요한 원본 로그 네 파일의 이름 목록입니다.
required = ['web_access.log','auth.log','firewall.log','network_flow.log']
assert all(os.path.exists(name) for name in required), '원본 로그 4개를 업로드하세요.'
print([name for name in os.listdir() if name.endswith('.log')])

`dict_keys(['web_access.log', ...])` 또는 원본 파일 4개의 목록이 출력되면 준비가 끝났습니다. CSV는 이후 셀에서 직접 만들어집니다.

## 실습 1. 웹 원본 로그 읽기

### 학습 목표
텍스트 한 줄이 HTTP 요청 한 건임을 확인한다.

### 배경 설명
웹 서버는 요청마다 문자열을 남긴다. CSV 열은 아직 존재하지 않는다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
import os, re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({'figure.figsize': (8, 4), 'axes.grid': False})
pd.set_option('display.max_columns', 15)
# web_lines: 웹 로그를 한 줄씩 담은 문자열 목록(list)입니다.
# 목록의 원소 한 개는 HTTP 요청 로그 한 줄이며, 아직 열로 분리된 표가 아닙니다.
# web_lines[0]은 첫 번째 로그입니다. Python 목록 인덱스는 0부터 시작합니다.
with open('web_access.log', encoding='utf-8') as f:
    web_lines = f.read().splitlines()
print('웹 로그 줄 수:', len(web_lines))
print(web_lines[0])

### 결과 확인 및 해석
800줄. 대괄호는 시간, 큰따옴표는 요청·Referrer·User-Agent를 감싼다.

### 생각해보기
한 줄 전체를 쉼표로 나누면 의미 있는 열이 될까?

## 실습 2. 인증 원본 로그 읽기

### 학습 목표
인증 성공·실패와 세션 메시지를 구분한다.

### 배경 설명
sshd는 인증 결과뿐 아니라 세션 시작과 종료도 기록한다. 모든 줄에 IP가 있는 것은 아니다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# auth_lines: SSH 인증·세션 로그 원문을 한 줄씩 담은 목록입니다.
# Accepted/Failed는 인증 결과, session opened/closed는 세션 상태를 나타냅니다.
# auth_lines[:5]는 처음 다섯 줄을 뜻하며, 일부 세션 줄에는 출발지 IP가 없습니다.
with open('auth.log', encoding='utf-8') as f:
    auth_lines = f.read().splitlines()
print('인증 로그 줄 수:', len(auth_lines))
for line in auth_lines[:5]:
    print(line)

### 결과 확인 및 해석
400줄. Accepted password/publickey, Failed password, Invalid user, session opened/closed가 포함된다.

### 생각해보기
session closed를 로그인 실패로 세면 어떤 오류가 생길까?

## 실습 3. 방화벽 원본 로그 읽기

### 학습 목표
KEY=VALUE 구조와 UFW 동작을 읽는다.

### 배경 설명
ALLOW는 방화벽 통과이고 BLOCK은 차단이다. 통과했다는 사실이 인증 성공을 뜻하지 않는다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# fw_lines: 방화벽 로그 원문 목록입니다. ALLOW/BLOCK은 방화벽의 허용/차단 결정입니다.
# SRC/DST는 출발지/목적지 IP, SPT/DPT는 출발지/목적지 포트, PROTO는 프로토콜입니다.
# IN/OUT은 입력/출력 인터페이스입니다. 빈 OUT=을 곧바로 수집 오류로 보지 않습니다.
with open('firewall.log', encoding='utf-8') as f:
    fw_lines = f.read().splitlines()
print('방화벽 로그 줄 수:', len(fw_lines))
print(fw_lines[0])

### 결과 확인 및 해석
700줄. ICMP에는 TCP/UDP 포트가 없고 TYPE/CODE가 있다.

### 생각해보기
차단한 통신도 보안 분석에 필요한가?

## 실습 4. Network Flow 원본 구조 확인

### 학습 목표
JSON Lines의 한 줄이 Flow 요약임을 이해한다.

### 배경 설명
Flow는 동일한 통신 특성으로 묶은 패킷 집합의 요약이다. 이 파일은 IPFIX 바이너리가 아닌 교육용 export 형식이다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# flow_lines: JSON 문자열 목록이며 한 줄은 패킷 한 개가 아니라 Flow 요약 한 건입니다.
# json.loads(...)는 JSON 문자열을 {필드명: 값} 사전으로 읽습니다.
# packets는 패킷 수, bytes는 이 실습의 단방향 L3 전송량(바이트), duration은 지속시간(초)입니다.
# timestamp의 +09:00은 한국 시간대 표시입니다. 여기에 9시간을 다시 더하지 않습니다.
with open('network_flow.log', encoding='utf-8') as f:
    flow_lines = f.read().splitlines()
print('Flow 수:', len(flow_lines))
print(json.loads(flow_lines[0]))

### 결과 확인 및 해석
1,100줄. bytes는 이 단방향 Flow의 관측 L3 전송량, duration은 초 단위이다.

### 생각해보기
웹 응답 본문 크기와 Flow bytes를 무조건 더해도 될까?

## 실습 5. 웹 로그 한 줄 파싱

### 학습 목표
문자열에서 필드를 추출한다.

### 배경 설명
정규표현식은 제공된 형식에만 맞춘다. 실제 운영에서는 escape, IPv6, 제품 버전을 추가 검증해야 한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# web_pattern: 웹 로그에서 다음 9개 필드를 추출하는 제공된 정규표현식입니다.
# src_ip=요청 출발지 IP, timestamp=시각, method=GET/POST 등의 HTTP 메서드,
# uri=요청 경로, http_version=HTTP 버전, status_code=HTTP 응답 상태 코드,
# bytes=응답 본문 크기(바이트), referrer=참조 페이지, user_agent=클라이언트 식별 문자열.
# (?P<필드명>...)은 추출할 값에 이름을 붙입니다. 숫자로 보이는 추출값도 이 단계에서는 문자열입니다.
web_pattern = re.compile(
    r'(?P<src_ip>\S+) \S+ \S+ \[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>\S+) (?P<uri>\S+) (?P<http_version>[^" ]+)" '
    r'(?P<status_code>\d{3}) (?P<bytes>\d+|-) '
    r'"(?P<referrer>[^"]*)" "(?P<user_agent>[^"]*)"'
)
# match: 한 줄 전체가 패턴과 맞으면 추출 결과 객체, 맞지 않으면 None입니다.
match = web_pattern.fullmatch(web_lines[0])
assert match is not None, '제공된 로그 형식과 패턴을 확인하세요.'
# groupdict(): 이름이 붙은 추출값을 사전으로 반환합니다. 예: {"src_ip": "192.168.10.25", ...}
print(match.groupdict())

### 결과 확인 및 해석
9개 필드가 dictionary로 출력된다. groupdict()는 이름이 붙은 추출값을 사전으로 반환한다.

### 생각해보기
공백으로만 split하면 시간과 User-Agent가 어떻게 나뉠까?

## 실습 6. 웹 전체를 DataFrame으로 변환

### 학습 목표
사전 목록을 행과 열로 바꾼다.

### 배경 설명
파싱 실패 줄을 조용히 버리지 않고 원본 줄 번호와 함께 보관한다. event_id는 수집기가 부여한 식별자이며 원본 필드는 아니다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# web_df: 웹 로그를 구조화한 DataFrame(행·열 표), web_errors: 파싱 실패 원문과 줄 번호 목록입니다.
# records는 정상 추출한 행 사전들의 목록, row는 로그 한 건의 필드 사전입니다.
def parse_web(lines):
    records, errors = [], []
    # line_no는 원본 파일의 실제 줄 번호이므로 1부터 셉니다. line은 해당 줄의 문자열입니다.
    for line_no, line in enumerate(lines, 1):
        m = web_pattern.fullmatch(line)
        if m is None:
            errors.append({'line_no': line_no, 'raw': line})
            continue
        row = m.groupdict()
        # raw_file + line_no로 원본을 찾아갑니다. CSV를 정렬해도 이 값은 바꾸지 않습니다.
        # event_id 예: web:00017. 이 실습에서 수집 이벤트를 구분하기 위해 추가한 식별자입니다.
        row.update(event_id=f'web:{line_no:05d}', line_no=line_no,
                   raw_file='web_access.log')
        records.append(row)
    return pd.DataFrame(records), errors
web_df, web_errors = parse_web(web_lines)
print(web_df.shape, '파싱 오류:', len(web_errors))
display(web_df.head())
assert len(web_df) == len(web_lines) and not web_errors

### 결과 확인 및 해석
800행. 원본에 있던 필드 9개와 추적 필드 3개를 구분한다.

### 생각해보기
왜 CSV에도 raw_file과 line_no를 남길까?

## 실습 7. 인증 로그 파싱

### 학습 목표
메시지 종류별로 동일한 필드를 추출한다.

### 배경 설명
제공된 함수를 실행하고 login_pattern의 다섯 값을 확인한다. session 메시지에는 IP·포트가 없으므로 억지로 채우지 않는다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# auth_prefix: 공통 머리글(timestamp/host/process/pid)과 본문(message)을 분리합니다.
# host=로그 발생 서버, process=sshd, pid=그 프로세스의 식별 번호입니다.
# login/invalid/session_pattern은 서로 다른 SSH 메시지 종류를 구분하는 패턴입니다.
auth_prefix = re.compile(r'(?P<timestamp>\w{3}\s+\d+ \d\d:\d\d:\d\d) '
                         r'(?P<host>\S+) (?P<process>sshd)\[(?P<pid>\d+)\]: (?P<message>.*)')
login_pattern = re.compile(r'(Accepted|Failed) (password|publickey) for '
                          r'(?:invalid user )?(\S+) from (\S+) port (\d+) ssh2')
invalid_pattern = re.compile(r'Invalid user (\S+) from (\S+) port (\d+)')
session_pattern = re.compile(r'pam_unix\(sshd:session\): session (opened|closed) for user (\S+)(?: by \(uid=0\))?')
def parse_auth(lines):
    records, errors = [], []
    for n, line in enumerate(lines, 1):
        m = auth_prefix.fullmatch(line)
        if not m:
            errors.append((n, line)); continue
        row = m.groupdict()
        msg = row.pop('message')
        login = login_pattern.fullmatch(msg)
        invalid = invalid_pattern.fullmatch(msg)
        session = session_pattern.fullmatch(msg)
        if login:
            # result=Accepted/Failed, method=password/publickey(인증 방식),
            # user=대상 계정, ip=출발지 IP, port=출발지 포트입니다. 여기서 method는 HTTP 메서드가 아닙니다.
            result, method, user, ip, port = login.groups()
            row.update(result=result, auth_method=method, username=user,
                       src_ip=ip, src_port=port)
        elif invalid:
            user, ip, port = invalid.groups()
            # Invalid는 존재하지 않는 계정에 대한 알림입니다. Failed와 같은 시도의 추가 기록일 수 있습니다.
            row.update(result='Invalid', username=user, src_ip=ip, src_port=port)
        elif session:
            state, user = session.groups()
            # 세션 메시지에 없는 IP·포트는 추측해서 채우지 않습니다. 표로 만들면 해당 칸은 결측이 됩니다.
            row.update(result='Session_' + state, username=user)
        else:
            errors.append((n, line)); continue
        row.update(event_id=f'auth:{n:05d}', line_no=n, raw_file='auth.log')
        records.append(row)
    return pd.DataFrame(records), errors
auth_df, auth_errors = parse_auth(auth_lines)
print(auth_df['result'].value_counts())
assert len(auth_df) == 400 and not auth_errors

### 결과 확인 및 해석
Failed 126, Accepted 82, Invalid 30, Session_opened 81, Session_closed 81. Invalid는 같은 실패 시도의 추가 메시지일 수 있다.

### 생각해보기
Failed와 Invalid를 합치면 로그인 시도 수를 중복 계산할 수 있는 이유는?

## 실습 8. 방화벽 KEY=VALUE 파싱

### 학습 목표
반복문과 split으로 필드를 분리한다.

### 배경 설명
split("=", 1)은 첫 번째 등호에서만 나눈다. OUT=의 빈 값은 정상적인 빈 인터페이스 값일 수 있다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# fw_prefix: 시각·호스트·방화벽 동작(action)·KEY=VALUE 본문(fields)을 분리합니다.
# fw_df는 파싱한 방화벽 표, fw_errors는 형식을 해석하지 못한 줄의 목록입니다.
fw_prefix = re.compile(r'(?P<timestamp>\w{3}\s+\d+ \d\d:\d\d:\d\d) '
                       r'(?P<host>\S+) kernel: \[UFW (?P<action>\w+)\] (?P<fields>.*)')
def parse_firewall(lines):
    records, errors = [], []
    for n, line in enumerate(lines, 1):
        m = fw_prefix.fullmatch(line)
        if not m:
            errors.append((n, line)); continue
        row = m.groupdict()
        # token은 'SRC=203.0.113.20' 같은 항목 하나입니다.
        # key는 SRC, value는 203.0.113.20이 됩니다. 원래 필드 이름은 정규화 전까지 유지합니다.
        for token in row.pop('fields').split():
            if '=' in token:
                key, value = token.split('=', 1)
                row[key] = value
        # ICMP의 TYPE/CODE는 메시지 종류/세부 코드입니다. TCP·UDP 포트인 SPT/DPT와 다릅니다.
        row.update(event_id=f'firewall:{n:05d}', line_no=n, raw_file='firewall.log')
        records.append(row)
    return pd.DataFrame(records), errors
fw_df, fw_errors = parse_firewall(fw_lines)
display(fw_df.head())
assert len(fw_df) == 700 and not fw_errors

### 결과 확인 및 해석
SRC, DST, PROTO, SPT, DPT 등 원래 이름을 우선 보존한다.

### 생각해보기
ICMP 행의 DPT 결측치는 수집 오류인가?

## 실습 9. Flow 구조화

### 학습 목표
JSON 사전 목록에서 DataFrame을 만든다.

### 배경 설명
이미 필드가 구조화된 JSON Lines는 복잡한 정규표현식이 필요 없다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# flow_records: JSON을 해석한 사전들의 목록, flow_df: 그 사전들을 행으로 쌓은 표입니다.
# src_ip/dst_ip와 src_port/dst_port는 출발지/목적지 주소·포트이며 protocol은 TCP/UDP/ICMP 등입니다.
# JSON의 null은 Python에서 None이 되고, DataFrame에서는 결측으로 다뤄집니다.
flow_records = [json.loads(line) for line in flow_lines]
flow_df = pd.DataFrame(flow_records)
# line_no/event_id/raw_file은 원문에 없던 추적 정보입니다. np.arange의 끝값은 포함되지 않습니다.
flow_df['line_no'] = np.arange(1, len(flow_df) + 1)
flow_df['event_id'] = flow_df['line_no'].map(lambda n: f'flow:{n:05d}')
flow_df['raw_file'] = 'network_flow.log'
display(flow_df.head())
assert len(flow_df) == 1100

### 결과 확인 및 해석
duration은 일부 null이다. JSON 숫자와 문자열의 자료형 차이를 확인한다.

### 생각해보기
구조화된 JSON에도 전처리가 필요한 이유는?

## 실습 10. CSV 저장과 교육용 수집 문제

### 학습 목표
원본 필드와 보강 필드의 출처를 구분하며 CSV를 저장한다.

### 배경 설명
아래 셀은 teaching_metadata.csv를 추가 업로드한다. 이 파일은 원본 로그가 아니라 합성 정답·보강값·수집 재전송 명세이다. label은 분석에 사용하지 않는다. country=LAB은 실재 국가가 아니다. replay=1인 원본 이벤트 45건을 수집기가 한 번 더 저장한 상황을 명시적으로 재현한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# meta: 합성 정답·서비스 보강값·교육용 수집 문제를 담은 보조 표입니다. 원본 로그가 아닙니다.
# label은 NORMAL/BRUTE_FORCE 등의 교수자 검증용 정답이며 의심 패턴을 찾는 입력으로 쓰지 않습니다.
# service는 서비스 분류, country=LAB는 합성 실습 표시로 실제 국가 정보가 아닙니다.
if not os.path.exists('teaching_metadata.csv'):
    from google.colab import files
    uploaded_metadata = files.upload()
assert os.path.exists('teaching_metadata.csv'), 'teaching_metadata.csv를 업로드하세요.'
meta = pd.read_csv('teaching_metadata.csv')
# parsed는 {출처명: 파싱한 DataFrame}, filenames는 {출처명: 저장할 CSV 파일명} 사전입니다.
parsed = {'web': web_df, 'auth': auth_df, 'firewall': fw_df, 'flow': flow_df}
filenames = {'web': 'web_access.csv', 'auth': 'auth.csv',
             'firewall': 'firewall.csv', 'flow': 'network_flow.csv'}
# dirty는 교육용 누락·자료형 문제·수집 중복을 포함하는 전처리 전 데이터 사전입니다.
# replay=1은 같은 이벤트의 수집 사본을 추가하라는 표시이며, 실제 반복 요청을 뜻하지 않습니다.
# bytes_override는 일부 Flow의 bytes를 천 단위 쉼표가 있는 문자열로 바꾸기 위한 보조 값입니다.
dirty = {}
for source, frame in parsed.items():
    cols = ['event_id', 'label', 'service', 'country', 'replay', 'bytes_override']
    # enriched: event_id로 원본 관측 필드에 보강값을 연결한 표입니다.
    # one_to_one은 양쪽 event_id가 각각 유일한지 검사하여 의도하지 않은 행 증식을 막습니다.
    enriched = frame.merge(meta[cols], on='event_id', validate='one_to_one')
    if source == 'flow':
        enriched['bytes'] = enriched['bytes'].astype('string')
        # mask는 bytes_override가 있는 행만 True인 조건 목록(불리언 Series)입니다.
        mask = enriched['bytes_override'].notna()
        enriched.loc[mask, 'bytes'] = enriched.loc[mask, 'bytes_override']
    duplicated_rows = enriched.loc[enriched['replay'].eq(1)]
    enriched = pd.concat([enriched, duplicated_rows], ignore_index=True)
    dirty[source] = enriched.drop(columns=['replay', 'bytes_override'])
    dirty[source].to_csv(filenames[source], index=False)
    print(filenames[source], dirty[source].shape)
# work는 dirty의 복사본입니다. 이후 work만 정제하여 전처리 전후를 비교할 수 있게 합니다.
work = {key: frame.copy() for key, frame in dirty.items()}

### 결과 확인 및 해석
web 812, auth 406, firewall 710, flow 1117행. 원본 3000건에 수집 중복 45건이 추가된다. CSV는 아직 정제 전이다.

### 생각해보기
동일한 줄 내용과 동일한 수집 event_id 중 무엇이 중복의 더 강한 근거인가?

## 실습 11. 데이터셋 컬럼과 기본 탐색

### 학습 목표
head/tail/shape/columns/info/describe 결과를 읽는다.

### 배경 설명
source별 관측 대상이 다르므로 컬럼이 다른 것이 정상이다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# source는 web/auth/firewall/flow 중 하나, frame은 해당 출처의 DataFrame입니다.
# shape=(행 수, 열 수), columns=열 이름 목록입니다. head/tail은 앞/뒤 행의 미리보기입니다.
# info는 자료형·비결측 수, describe는 요약 통계입니다. 문자열 bytes에는 수치 평균이 나오지 않습니다.
for source, frame in dirty.items():
    print(source, frame.shape, list(frame.columns))
print(pd.read_csv('web_access.csv').head(2))
display(dirty['flow'].tail(2))
dirty['flow'].info()
display(dirty['flow'].describe(include='all'))

### 결과 확인 및 해석
행 수와 열 수는 shape, 열 이름은 columns, 비결측 개수와 자료형은 info로 확인한다. 숫자 통계는 숫자로 변환한 뒤 다시 보자.

### 생각해보기
bytes가 문자열이면 describe는 평균 대신 무엇을 보여주는가?

## 실습 12. 결측치 확인

### 학습 목표
표현상 결측치와 의미상 결측치를 구분한다.

### 배경 설명
NaN/None/pd.NA, 빈 문자열, 원본의 -가 모두 결측 표현이 될 수 있다. auth session IP와 ICMP 포트는 원래 해당 필드가 없다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# 빈 문자열과 '-'를 pandas 결측 표시 pd.NA로 통일합니다.
# isna().sum()은 컬럼별 결측 개수입니다. lambda x: x > 0은 결측이 있는 컬럼만 선택합니다.
# 원래 없는 필드(세션 IP·ICMP 포트 등)와 실제 누락을 구분해야 하므로 행을 일괄 삭제하지 않습니다.
for source, frame in work.items():
    frame.replace({'': pd.NA, '-': pd.NA}, inplace=True)
    print(source)
    print(frame.isna().sum()[lambda x: x > 0])

### 결과 확인 및 해석
의도한 오류성 결측은 고유 이벤트 기준 90건(3%): UA 24, duration 33, service 10, country 23. 구조적 결측은 별도로 본다.

### 생각해보기
결측 비율이 높은 auth.src_ip 행을 모두 삭제하면 어떤 기록을 잃을까?

## 실습 13. 결측치 처리

### 학습 목표
필드별 정책을 적용한다.

### 배경 설명
service/country/user_agent는 미상으로 표시한다. duration을 평균이나 0으로 채우면 속도 해석이 왜곡되므로 결측을 남기고 표시한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# UNKNOWN은 범주 값을 모른다는 문자열 표시입니다. 정상/공격의 의미를 갖지 않습니다.
# duration_missing은 지속시간이 비어 있으면 True인 표시 열이며 Flow에만 추가합니다.
# duration의 빈칸 자체는 유지합니다. 0초로 채우면 실제로 즉시 끝난 통신과 구분할 수 없습니다.
for source, frame in work.items():
    frame['service'] = frame['service'].fillna('UNKNOWN')
    frame['country'] = frame['country'].fillna('UNKNOWN')
    if source == 'web':
        frame['user_agent'] = frame['user_agent'].fillna('UNKNOWN')
    if source == 'flow':
        frame['duration_missing'] = frame['duration'].isna()
print(work['flow']['duration_missing'].sum())

### 결과 확인 및 해석
duration 결측 33개는 보존된다. 정제 완료는 모든 칸이 채워졌다는 뜻이 아니다.

### 생각해보기
이 값이 이상치라면 삭제해야 하는가? 결측과 이상치는 어떻게 다른가?

## 실습 14. 중복과 반복 이벤트

### 학습 목표
수집 event_id로 재전송 중복만 제거한다.

### 배경 설명
event_id는 파일 식별자와 줄 번호로 만든다. 이 실습의 메타데이터가 같은 ID 재전송을 보장한다. 실제 운영에서는 로그 회전·수집기·장비 ID까지 포함한 키가 필요하다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# event_id가 같은 추가 사본만 중복으로 셉니다. 같은 IP가 반복 등장하는 것은 중복 기준이 아닙니다.
# 서로 다른 원본 줄에서 발생한 실제 로그인 실패는 서로 다른 event_id이므로 보존됩니다.
for source, frame in work.items():
    print(source, '중복:', frame.duplicated('event_id').sum())
    work[source] = frame.drop_duplicates('event_id').copy()
assert sum(len(frame) for frame in work.values()) == 3000

### 결과 확인 및 해석
중복 web 12, auth 6, firewall 10, flow 17. 90개의 brute-force Failed 이벤트는 서로 다른 ID라 그대로 보존된다.

### 생각해보기
src_ip와 uri만으로 drop_duplicates하면 공격 패턴에 어떤 일이 생길까?

## 실습 15. 자료형 확인과 변경

### 학습 목표
문자열 숫자를 수치형으로 바꾸고 실패를 확인한다.

### 배경 설명
Flow 12행의 bytes에 교육용 천 단위 쉼표가 포함되었다. 단위를 알고 있으므로 쉼표만 제거한다. errors=coerce는 실패를 NaN으로 만들므로 전후 결측 수를 확인한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# numeric은 수치형으로 바꿀 후보 컬럼 이름 목록입니다. 출처에 존재하는 컬럼만 처리합니다.
# before는 변환 전 값이 있던 칸 수, values는 쉼표를 제거한 임시 문자열 Series입니다.
# coerce가 변환 불가 값을 결측으로 바꿀 수 있어, 변환 전후 비결측 개수가 같은지 검사합니다.
# Int64(대문자 I)는 결측을 허용하는 정수형입니다. duration은 소수 초가 가능하므로 정수화하지 않습니다.
numeric = ['bytes', 'packets', 'duration', 'src_port', 'dst_port',
           'SPT', 'DPT', 'status_code', 'pid']
for source, frame in work.items():
    for col in numeric:
        if col in frame:
            before = frame[col].notna().sum()
            values = frame[col].astype('string').str.replace(',', '', regex=False)
            frame[col] = pd.to_numeric(values, errors='coerce')
            assert frame[col].notna().sum() == before
            if col != 'duration':
                frame[col] = frame[col].astype('Int64')
            assert frame[col].dropna().ge(0).all()
print(work['flow'].dtypes)

### 결과 확인 및 해석
Int64는 결측을 허용하는 정수형이다. 포트는 0~65535, duration·bytes는 0 이상인지 추가 검사한다.

### 생각해보기
900 MB Flow는 파싱 오류인가, 실제 관측값인가? 원본에서 확인하자.

## 실습 16. 시간 정보 변환

### 학습 목표
서로 다른 문자열 시간을 동일한 시간대의 datetime으로 바꾼다.

### 배경 설명
실습은 2026-09-16, 모든 장비 Asia/Seoul, 시계 동기화 가정이다. 실제 syslog의 연도·시간대는 수집 설정에서 확인해야 하며 현재 연도를 임의로 붙이지 않는다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# series는 한 출처의 timestamp 열, source는 그 로그의 종류입니다.
# tz_localize는 시간대 없는 시각에 시간대 의미를 부여하고, tz_convert는 같은 순간의 표시 시간대를 바꿉니다.
# Web의 +0900, Flow의 +09:00은 이미 시각에 포함된 UTC 오프셋입니다. 수동으로 9시간을 더하지 않습니다.
# Auth/Firewall의 2026년·한국 시간대는 원문이 아니라 수집 환경 명세에서 보강한 정보입니다.
def convert_time(series, source):
    if isinstance(series.dtype, pd.DatetimeTZDtype):
        return series.dt.tz_convert('Asia/Seoul')
    if source == 'web':
        return pd.to_datetime(series, format='%d/%b/%Y:%H:%M:%S %z', utc=True).dt.tz_convert('Asia/Seoul')
    if source in ['auth', 'firewall']:
        # 원본 syslog에 없는 연도와 시간대를 수집 환경 명세에서 보충한다.
        values = '2026 ' + series.astype(str)
        return pd.to_datetime(values, format='%Y %b %d %H:%M:%S').dt.tz_localize('Asia/Seoul')
    return pd.to_datetime(series, utc=True).dt.tz_convert('Asia/Seoul')
for source, frame in work.items():
    frame['timestamp'] = convert_time(frame['timestamp'], source)
print(work['auth']['timestamp'].head())

### 결과 확인 및 해석
모든 시간은 +09:00. dt.hour는 서울 현지 시간 기준이다. 연말 로그와 UTC 로그가 섞인 상황에서는 추가 정책이 필요하다.

### 생각해보기
UTC 18:12와 다음 날 KST 03:12는 서로 다른 사건인가?

## 실습 17. 범주형과 One-Hot Encoding

### 학습 목표
범주별 개수와 표시 열을 만든다.

### 배경 설명
TCP=0, UDP=1, ICMP=2는 크기 순서가 아니다. One-Hot은 범주를 별도 열로 만든다. 이번 주에는 모델을 학습하지 않는다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# protocol은 통신 프로토콜의 범주입니다. 숫자를 붙이더라도 크기·위험 순서를 뜻하지 않습니다.
# demo는 One-Hot Encoding 설명용 작은 표이며 실제 Flow를 덮어쓰지 않습니다.
# get_dummies는 범주별 열을 만들어 해당 범주이면 1, 아니면 0으로 표시합니다.
print(work['flow']['protocol'].value_counts(dropna=False))
demo = pd.DataFrame({'protocol': ['TCP', 'UDP', 'ICMP']})
display(pd.get_dummies(demo, columns=['protocol'], dtype=int))

### 결과 확인 및 해석
TCP 행은 protocol_TCP=1, 다른 열은 0. 미지 범주는 이후 학습/추론 단계에서 같은 규칙으로 다뤄야 한다.

### 생각해보기
IP 주소를 단순 숫자로 바꿔 모델에 넣으면 무엇을 잘못 학습할 수 있는가?

## 실습 18. 공통 컬럼 정의

### 학습 목표
분석 질문에 맞는 최소 공통 스키마를 정한다.

### 배경 설명
label은 합성 정답이며 관측 필드가 아니다. bytes_scope로 측정 의미를 보존한다. 적용 불가 값은 NaN으로 둔다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# COMMON: 서로 다른 출처를 통합할 때 사용할 공통 컬럼 이름·순서입니다.
# timestamp=시각, source_type=출처, src_ip/dst_ip=출발지/목적지 IP, src_port/dst_port=각 포트.
# protocol=전송 프로토콜, action=출처별 행위·결과, event_type=이벤트 종류.
# bytes=바이트 수, packets=패킷 수, duration=초, bytes_scope=bytes의 측정 범위.
# username=계정, result=원래 SSH 결과, method=HTTP 메서드, uri=요청 경로,
# status_code=HTTP 응답 코드, user_agent=클라이언트 식별 문자열.
# event_id/raw_file/line_no=원본 추적 정보, label=합성 정답, service/country=교육용 보강값,
# duration_missing=Flow 지속시간 누락 표시. 해당 출처에 원래 없는 필드는 결측으로 남습니다.
COMMON = ['timestamp', 'source_type', 'src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'action', 'event_type', 'bytes', 'username', 'uri', 'status_code', 'label', 'event_id', 'raw_file', 'line_no', 'packets', 'duration', 'result', 'method', 'user_agent', 'service', 'country', 'bytes_scope', 'duration_missing']
print(COMMON)

### 결과 확인 및 해석
기본 14개 공통 필드에 출처·단위·추적·실습용 필드를 포함해 26열이다.

### 생각해보기
왜 Raw Log를 처음부터 한 CSV에 섞지 않았는가?

## 실습 19. source_type 추가

### 학습 목표
행이 어느 시스템에서 생성되었는지 보존한다.

### 배경 설명
같은 IP가 여러 시스템에 나타나도 로그의 의미는 서로 다르다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# source_type은 각 행이 어느 종류의 로그에서 왔는지를 나타냅니다.
# 예: auth는 SSH 로그, flow는 네트워크 요약입니다. 장비의 이름이나 IP를 뜻하지 않습니다.
for source, frame in work.items():
    frame['source_type'] = source
print(work['auth'][['source_type', 'timestamp', 'result']].head())

### 결과 확인 및 해석
web/auth/firewall/flow의 네 가지 값이다. 통합 행 수는 네트워크 연결 수와 같지 않다.

### 생각해보기
동일한 SSH 시도가 firewall·auth·flow에 기록되면 사건 수는 몇 개인가?

## 실습 20. 컬럼명과 의미 정규화

### 학습 목표
SRC를 src_ip로, BLOCK을 DENY로 매핑한다.

### 배경 설명
HTTP 401은 인증 필요/실패 응답이며 방화벽 DENY와 같은 사건이 아니다. action=REQUEST를 유지하고 status_code로 분석한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# normalize는 컬럼 이름·시간·동작 값을 공통 스키마에 맞춥니다. x는 입력 표의 복사본입니다.
# BLOCK→DENY는 같은 방화벽 차단 의미를 통일한 것입니다. HTTP 상태 코드를 DENY로 바꾸지는 않습니다.
# action의 REQUEST/OBSERVE는 각각 웹 요청/Flow 관측이며 정상·공격 판정값이 아닙니다.
# bytes_scope로 HTTP 응답 본문(HTTP_RESPONSE_BODY)과 L3 Flow 전송량(FLOW_L3_TOTAL)을 구분합니다.
def normalize(frame, source):
    x = frame.copy()
    x['source_type'] = source
    x['timestamp'] = convert_time(x['timestamp'], source)
    if source == 'firewall':
        x = x.rename(columns={'SRC': 'src_ip', 'DST': 'dst_ip',
                              'SPT': 'src_port', 'DPT': 'dst_port', 'PROTO': 'protocol'})
        x['action'] = x['action'].replace({'BLOCK': 'DENY'})
        x['event_type'] = 'FIREWALL_DECISION'
    elif source == 'web':
        # 서버 배치 명세에서 얻은 값. access.log에서 추출한 값은 아니다.
        x['dst_ip'], x['dst_port'], x['protocol'] = '192.168.10.20', 80, 'TCP'
        x['action'], x['event_type'] = 'REQUEST', 'HTTP_REQUEST'
        x['bytes_scope'] = 'HTTP_RESPONSE_BODY'
    elif source == 'auth':
        # SSH 목적지 서버·22번 포트·TCP도 실습 배치 명세의 보강값이며 auth 원문에서 추출한 값이 아닙니다.
        x['dst_ip'], x['dst_port'], x['protocol'] = '192.168.10.10', 22, 'TCP'
        x['action'] = x['result'].map({'Failed': 'FAIL', 'Accepted': 'SUCCESS',
            'Invalid': 'INVALID_USER', 'Session_opened': 'SESSION_OPEN',
            'Session_closed': 'SESSION_CLOSE'})
        x['event_type'] = np.where(x['result'].str.startswith('Session'), 'SSH_SESSION', 'SSH_AUTH')
    else:
        x['action'], x['event_type'] = 'OBSERVE', 'NETWORK_FLOW'
        x['bytes_scope'] = 'FLOW_L3_TOTAL'
    # 공통 열 순서에 맞추고 없는 열은 결측으로 추가합니다. COMMON에 없는 원본 전용 열은 통합본에서 빠집니다.
    return x.reindex(columns=COMMON)
display(normalize(work['firewall'], 'firewall').head())

### 결과 확인 및 해석
방화벽의 대문자 원본 이름이 공통 소문자 이름으로 바뀐다. 추적 필드로 원본을 다시 찾을 수 있다.

### 생각해보기
이름이 같은 bytes라도 의미가 다르면 어떻게 해야 하는가?

## 실습 21. concat으로 통합하고 정제본 저장

### 학습 목표
정규화된 행들을 쌓고 원본 추적성을 확인한다.

### 배경 설명
통합 전 단계의 dirty는 결측/중복을 유지한다. work는 필드별 정제를 끝냈다. 두 결과를 각각 저장한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# integrated는 dirty에서 만든 전처리 전 통합본(3,045행), df는 work에서 만든 정제 통합본(3,000행)입니다.
# concat은 서로 다른 출처의 기록을 세로로 쌓습니다. 같은 연결의 기록을 한 사건으로 병합하는 것은 아닙니다.
integrated = pd.concat([normalize(v, k) for k, v in dirty.items()], ignore_index=True)
df = pd.concat([normalize(v, k) for k, v in work.items()], ignore_index=True)
df = df.sort_values(['timestamp', 'event_id']).reset_index(drop=True)
# hour/minute는 한국 시간 기준 시·분, weekday는 월요일=0부터 일요일=6까지입니다.
# business_hours는 이 실습에서 09:00~17:59이면 True입니다. 주말 여부를 함께 검사하는 규칙은 아닙니다.
df['hour'] = df['timestamp'].dt.hour
df['minute'] = df['timestamp'].dt.minute
df['weekday'] = df['timestamp'].dt.dayofweek
df['business_hours'] = df['hour'].between(9, 17)
integrated.to_csv('security_events_integrated.csv', index=False)
df.to_csv('security_events_clean.csv', index=False)
assert integrated.shape == (3045, 26)
assert df.shape == (3000, 30) and df['event_id'].is_unique
# 이 줄부터 네 *_df 변수는 최초 파싱 표가 아니라 정제 통합본에서 출처별로 다시 나눈 표입니다.
web_df, auth_df, fw_df, flow_df = [df[df['source_type'].eq(s)].copy()
                                 for s in ['web', 'auth', 'firewall', 'flow']]
print(integrated.shape, df.shape)
display(df['source_type'].value_counts())
df['source_type'].value_counts().plot.bar(color='#375B88', ylabel='Events')
plt.title('Events by source (clean)'); plt.tight_layout(); plt.show()

### 결과 확인 및 해석
3045행 26열 → 3000행 30열. clean에도 구조적 결측과 duration 결측은 남는다. 관찰: Flow가 가장 많다. 가능한 원인: 수집 범위 차이. 추가 확인: 장비별 수집 정책.

### 생각해보기
여러 로그를 왜 통합해야 하는가? concat과 join의 차이는?

## 실습 22. Source IP별 이벤트 수

### 학습 목표
groupby와 size로 행 수를 센다.

### 배경 설명
통합 이벤트 수가 많다고 연결이 많은 것은 아니다. 출처별 개수도 함께 본다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# ip_counts: 인덱스는 src_ip, 값은 해당 IP가 등장한 통합 로그의 행 수인 Series입니다.
# 사용자 수·연결 수·공격 횟수와는 다릅니다. 같은 통신이 여러 출처에 기록될 수 있습니다.
# groupby는 기본적으로 src_ip가 없는 행을 집계에서 제외합니다. 웹 요청만 셀 때는 web_df를 사용합니다.
ip_counts = df.groupby('src_ip').size().sort_values(ascending=False)
display(ip_counts.head(10))
display(web_df.groupby('src_ip').size().nlargest(10))
print('IP가 없는 행:', df['src_ip'].isna().sum())

### 결과 확인 및 해석
groupby는 기본적으로 결측 키를 제외한다. auth 세션 행 162개는 IP별 집계에 포함되지 않는다.

### 생각해보기
정상 모니터링 서버 192.168.10.5의 요청 증가를 공격이라고 할 수 있는가?

## 실습 23. Destination Port 분석

### 학습 목표
isin과 Top N 집계를 사용한다.

### 배경 설명
22/23/3389는 점검할 포트 예시일 뿐 특정 포트 사용 자체가 공격은 아니다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# selected_ports_answer는 학생이 작성할 필터 결과(DataFrame)를 담는 자리입니다.
# port_top은 목적지 포트별 기록 수의 상위 10개입니다. IP나 패킷 수가 아니라 로그 행 수를 셉니다.
# 목적지 포트가 없는 행은 이 집계에서만 제외하며 전체 df에서 삭제하지 않습니다.
# 문제 2. 목적지 포트가 22, 23, 3389인 행을 추출하세요.
selected_ports_answer = None  # TODO: df[...] 식으로 바꾸세요.
if selected_ports_answer is not None:
    display(selected_ports_answer.head())
# 다음 실습은 이 답에 의존하지 않습니다.
port_top = df['dst_port'].dropna().astype(int).value_counts().head(10)
port_top.plot.bar(color='#375B88', ylabel='Events', xlabel='Destination port')
plt.title('Top destination ports'); plt.tight_layout(); plt.show()

### 결과 확인 및 해석
관찰: 22번 이벤트가 많다. 가능한 원인: 정상 관리와 반복 로그인. 추가 확인: auth 결과·IP·시간 범위.

### 생각해보기
ICMP 포트를 0으로 채워 집계하면 어떤 가짜 범주가 생길까?

## 실습 24. 로그인 실패 분석

### 학습 목표
Failed만 골라 IP별 실패 횟수를 센다.

### 배경 설명
Invalid user 및 session 기록을 로그인 실패 횟수에 더하지 않는다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# failed는 SSH 결과가 정확히 Failed인 행만 모은 표입니다. Invalid·세션 메시지는 제외합니다.
# failure_counts는 출발지 IP별 인증 실패 기록 수입니다. 실패가 집중된 시간도 함께 봐야 합니다.
failed = auth_df[auth_df['result'].eq('Failed')]
failure_counts = failed.groupby('src_ip').size().sort_values(ascending=False)
display(failure_counts.head(10))
# 문제 1. 실패 횟수가 5회 이상인 IP를 선택하세요.
failed_5_answer = None  # TODO: failure_counts[...] 식으로 바꾸세요.
if failed_5_answer is not None:
    display(failed_5_answer)

### 결과 확인 및 해석
203.0.113.25: 90회, 192.168.10.40: 36회. 후자는 5시간 15분 동안 간헐적 실패이다.

### 생각해보기
5회 이상이라는 조건만으로 공격을 확정하면 무엇을 놓치는가?

## 실습 25. DENY 이벤트 분석

### 학습 목표
방화벽 차단 기록의 상위 IP를 찾는다.

### 배경 설명
원본 BLOCK은 공통 스키마의 DENY로 바뀌었다. 분모는 방화벽 기록만 사용한다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# deny는 방화벽 기록 중 정규화된 action이 DENY인 행만 모은 표입니다.
# actions는 ALLOW/DENY별 기록 수이며, 백분율의 분모는 방화벽 행 수입니다.
deny = fw_df[fw_df['action'].eq('DENY')]
# 문제 3. 차단 이벤트가 많은 IP 상위 5개를 구하세요.
deny_top5_answer = None  # TODO: groupby, size, nlargest를 사용하세요.
if deny_top5_answer is not None:
    display(deny_top5_answer)
actions = fw_df['action'].value_counts().reindex(['ALLOW','DENY'])
(actions / actions.sum() * 100).plot.bar(color=['#375B88','#D66B45'], ylabel='Percent')
plt.title('Firewall decisions (%)'); plt.tight_layout(); plt.show()

### 결과 확인 및 해석
ALLOW 520, DENY 180. DENY Top은 203.0.113.20(120), 198.51.100.90(60). 해당 IP는 두 개뿐이므로 Top 5가 5행일 필요는 없다.

### 생각해보기
차단율이 높다는 사실은 방어 성공과 공격 증가 중 어느 쪽을 뜻하는가?

## 실습 26. 시간대별 이벤트와 인증 실패

### 학습 목표
현지 시각의 시간별 개수를 비교한다.

### 배경 설명
하루 중 시간대별 전체 이벤트와 인증 실패를 구분한다. 여러 날 데이터는 resample로 날짜도 보존한다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# hourly는 0~23시별 전체 로그 행 수입니다. 기록이 없는 시간은 0으로 채워 그래프 축을 유지합니다.
# fail_hour_answer는 전체 이벤트가 아닌 인증 실패만의 시간별 개수를 담을 학생 답안 자리입니다.
hourly = df.groupby('hour').size().reindex(range(24), fill_value=0)
hourly.plot(marker='o', color='#375B88', ylabel='Events', xlabel='Hour (KST)')
plt.title('Hourly events'); plt.tight_layout(); plt.show()
# 문제 4. failed의 timestamp에서 hour를 꺼내 시간대별 실패 횟수를 그리세요.
fail_hour_answer = None  # TODO: groupby(...).size().reindex(...) 완성
if fail_hour_answer is not None:
    fail_hour_answer.plot.bar(ylabel='Failed logins', xlabel='Hour (KST)')
    plt.tight_layout(); plt.show()

### 결과 확인 및 해석
관찰: 10시에 실패가 집중된다. 가능한 원인: 재시도·자동화. 추가 확인: 5분 구간과 계정·성공 기록.

### 생각해보기
시간대가 다르거나 장비 시계가 틀리면 상관분석이 어떻게 달라질까?

## 실습 27. Protocol 분포와 평균 전송량

### 학습 목표
Flow만 선택해 프로토콜별 빈도·평균을 계산한다.

### 배경 설명
통합 데이터에서는 TCP 한 연결을 여러 시스템이 관찰할 수 있다. 연결 특성은 Flow 자료로 범위를 제한한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# protocol_counts는 Flow만 대상으로 센 TCP/UDP/ICMP별 건수입니다.
# 아래 평균 bytes도 Flow끼리 계산합니다. Web bytes와 섞으면 측정 범위가 달라 해석이 왜곡됩니다.
# df의 action/event_type 분포는 전체 출처를 포함하므로 protocol_counts와 집계 대상이 다릅니다.
protocol_counts = flow_df['protocol'].value_counts()
protocol_counts.plot.bar(color='#375B88', ylabel='Flows')
plt.title('Protocols in network flow'); plt.tight_layout(); plt.show()
display(flow_df.groupby('protocol')['bytes'].mean())
display(df['action'].value_counts())
display(df['event_type'].value_counts())

### 결과 확인 및 해석
관찰: TCP가 다수이다. 가능한 원인: 웹·SSH·스캔·전송. 추가 확인: 목적지·포트·분포. 평균은 대용량 소수 이벤트의 영향을 받는다.

### 생각해보기
평균 bytes가 크다는 사실만으로 해당 프로토콜이 위험한가?

## 실습 28. Source IP Top 10 시각화

### 학습 목표
긴 IP 라벨을 가로 막대로 비교한다.

### 배경 설명
그림의 분모는 전체 로그 행이며 실제 사용자 수나 공격 횟수가 아니다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# 전체 출처의 src_ip 기록 수 Top 10입니다. 동률 IP가 있으면 경계에서 선택되는 IP가 달라질 수 있습니다.
# 마지막 sort_values는 가로 막대에서 큰 값이 위로 오도록 표시 순서를 정할 뿐 기록 수를 바꾸지 않습니다.
df['src_ip'].value_counts().head(10).sort_values().plot.barh(color='#375B88', xlabel='Events')
plt.title('Top source IPs (all sources)'); plt.tight_layout(); plt.show()

### 결과 확인 및 해석
관찰: 일부 외부 주소에 이벤트가 집중된다. 가능한 원인: 자동화·관리·스캐너. 추가 확인: source_type별 개수, 실패·차단·시간.

### 생각해보기
이 이벤트가 반드시 공격이라고 할 수 있는가? 정상 행위일 가능성은 없는가?

## 실습 29. Bytes 분포와 큰 연결 Top 10

### 학습 목표
대용량 꼬리를 보존하며 분포를 본다.

### 배경 설명
웹 bytes와 Flow bytes는 의미가 다르므로 Flow만 사용한다. log10(bytes+1)은 분포를 읽기 위한 표시 변환이며 원본을 바꾸지 않는다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# bytes는 Flow 전송량(바이트)입니다. log10(bytes+1)은 큰 범위의 값을 읽기 쉽게 표시하는 변환입니다.
# 예: 9 부근은 약 10억 바이트입니다. 원본 bytes 열은 그대로 유지합니다.
# bins는 로그 변환된 축에서 2~10.5 사이를 0.5 간격으로 나눈 구간 경계입니다.
plt.hist(np.log10(flow_df['bytes'].astype(float) + 1), bins=np.arange(2, 10.51, .5), color='#375B88')
plt.xlabel('log10(bytes + 1)'); plt.ylabel('Flows')
plt.title('Flow byte distribution'); plt.tight_layout(); plt.show()
# 문제 5. bytes가 가장 큰 Flow 10개를 추출하세요.
largest_flows_answer = None  # TODO: nlargest 사용
if largest_flows_answer is not None:
    display(largest_flows_answer[['src_ip','dst_ip','bytes']])

### 결과 확인 및 해석
관찰: 9 부근은 약 10억 bytes. 가능한 원인: 백업·대용량 전송. 추가 확인: 목적지와 승인 작업. 정상 내부 백업도 포함된다.

### 생각해보기
이 값이 이상치라면 삭제해야 하는가?

## 실습 30. Bytes와 Duration 산점도

### 학습 목표
두 수치 필드의 관계를 읽는다.

### 배경 설명
duration 결측 33행은 이 그림에서만 제외한다. 전체 정제본에서는 보존한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# scatter는 bytes와 duration이 모두 있어 산점도에 표시할 수 있는 Flow만 담은 임시 표입니다.
# 점 하나는 Flow 한 건이며 x축은 지속시간(초), y축은 log10(전송 바이트+1)입니다.
# duration이 없는 행은 이 그림에서만 제외하고 flow_df와 전체 정제 데이터에는 남겨 둡니다.
scatter = flow_df.dropna(subset=['bytes','duration'])
plt.scatter(scatter['duration'], np.log10(scatter['bytes'].astype(float) + 1), s=15, alpha=.5, color='#375B88')
plt.xlabel('Duration (seconds)'); plt.ylabel('log10(bytes + 1)')
plt.title('Bytes vs duration'); plt.tight_layout(); plt.show()
print('산점도 포함:', len(scatter), '/ 전체:', len(flow_df))

### 결과 확인 및 해석
1067개 점. 관찰: 짧고 작은 Flow와 대용량 Flow가 나뉜다. 가능한 원인: 서비스 차이. 추가 확인: 포트·목적지·패킷 수.

### 생각해보기
그림에서 제외한 행을 전체 데이터에서도 삭제해야 하는가?

## 실습 31. Brute Force 의심 후보

### 학습 목표
5분 구간 안의 반복 실패와 이후 성공을 확인한다.

### 배경 설명
일별 횟수만 보면 정상 사용자의 간헐적 실패를 과대평가할 수 있다. 아래 임계값은 이 실습용 후보 탐색 규칙이다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# failed_window는 인증 실패 표의 복사본입니다. window는 실패 시각을 5분 구간 시작으로 내린 값입니다.
# 예: 10:22:03→10:20:00. burst는 (출발지 IP, 5분 구간)별 실패 기록 수입니다.
# 5회 이상은 실습용 조사 후보 기준이며 공격 확정 기준이 아닙니다. 고정 구간 경계에 걸친 패턴은 놓칠 수 있습니다.
failed_window = failed.copy()
failed_window['window'] = failed_window['timestamp'].dt.floor('5min')
burst = failed_window.groupby(['src_ip','window']).size().rename('failures')
display(burst[burst >= 5])
display(auth_df[(auth_df['src_ip'].eq('203.0.113.25')) &
                (auth_df['result'].eq('Accepted'))][['timestamp','username','result']])

### 결과 확인 및 해석
203.0.113.25가 10:20~10:24:59 구간에 90회 실패하고 10:24:40에 성공한다. 성공은 침해 확정 증거가 아니다.

### 생각해보기
고정 구간의 경계에 걸친 공격은 어떻게 놓칠 수 있는가? 슬라이딩 윈도는 다음 단계로 남긴다.

## 실습 32. Port Scan 의심 후보

### 학습 목표
짧은 시간에 접근한 서로 다른 목적지 포트 수를 센다.

### 배경 설명
이 실습의 Flow 센서는 방화벽 외부에 있어 차단된 시도도 관찰한다. 내부 센서라면 같은 기록을 기대하면 안 된다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# scan_work는 방화벽 정제본의 복사본, ports는 (출발지 IP, 5분 구간)별 서로 다른 목적지 포트 수입니다.
# nunique는 고유 값 수이므로 같은 포트로 100번 접근해도 1개로 셉니다.
# flow_destinations는 Flow에서 출발지별로 접근한 서로 다른 목적지 IP 수입니다.
# 포트 다양성은 수직 탐색, 목적지 다양성은 수평 탐색을 살펴볼 단서이며 승인된 점검일 수도 있습니다.
scan_work = fw_df.copy()
scan_work['window'] = scan_work['timestamp'].dt.floor('5min')
ports = scan_work.groupby(['src_ip','window'])['dst_port'].nunique()
display(ports[ports >= 20])
flow_destinations = flow_df.groupby('src_ip')['dst_ip'].nunique().sort_values(ascending=False)
display(flow_destinations.head())

### 결과 확인 및 해석
203.0.113.20이 11:00~11:04:59에 120포트. 198.51.100.60은 13시에 30개 목적지로 445 접근.

### 생각해보기
승인된 취약점 점검과 비인가 스캔을 구별하려면 어떤 추가 로그를 확인해야 하는가?

## 실습 33. 대용량 전송 후보

### 학습 목표
같은 임계값을 넘는 정상·의심 사례를 비교한다.

### 배경 설명
500 MB는 실습용 탐색 기준이며 보편적 공격 판정값이 아니다. bytes가 큰 행은 삭제하지 않고 별도로 조사한다.

### 코드
위에서 아래로 실행하세요.

In [ ]:
# large는 전송량이 500,000,000바이트(십진 기준 500MB) 이상인 Flow 후보 표입니다.
# 정상 백업과 의심 외부 전송이 함께 포함됩니다. 임계값을 넘었다고 삭제하거나 공격으로 확정하지 않습니다.
large = flow_df[flow_df['bytes'] >= 500_000_000]
display(large[['timestamp','src_ip','dst_ip','dst_port','bytes','duration']])

### 결과 확인 및 해석
8건: 외부 8443 목적지 5건과 내부 백업 3건. IP 주소 대역 자체로 실제 소유자나 국가를 추정하지 않는다.

### 생각해보기
백업 일정, 전송 승인, Endpoint 프로세스 중 무엇을 추가 확인할까?

## 실습 34. 여러 로그에서 같은 IP 확인

### 학습 목표
교집합과 시간순 정렬로 맥락을 연결한다.

### 배경 설명
같은 IP라도 NAT나 공유 주소일 수 있다. IP만 같은 사실보다 시간·목적지·포트·계정을 함께 보는 것이 낫다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# common_ips_answer는 두 출처에 모두 나타난 IP의 집합(set)을 담을 학생 답안 자리입니다.
# timeline은 대표 IP와 관련된 여러 출처의 기록을 시각순으로 나열한 표입니다.
# 같은 IP의 시간적 연속성을 보여줄 뿐, 동일한 사람이나 동일한 사건임을 자동으로 증명하지는 않습니다.
# 문제 6. 방화벽과 인증 로그에 모두 등장하는 src_ip를 찾으세요.
common_ips_answer = None  # TODO: set(...) & set(...)
if common_ips_answer is not None:
    print(sorted(common_ips_answer))
timeline = df[df['src_ip'].eq('203.0.113.25')].sort_values(['timestamp','source_type'])
display(timeline[['timestamp','source_type','src_ip','src_port','dst_port','action','username']].head(12))

### 결과 확인 및 해석
10:20:01 firewall ALLOW와 Flow, 10:20:02 auth Failed의 연결을 확인한다. 통합 조회는 편리하지만 사건 병합을 자동으로 증명하지 않는다.

### 생각해보기
서로 다른 로그를 왜 통합해야 하는가? 같은 source IP만으로 같은 사람이라고 할 수 있는가?

## 실습 35. 종합 문제와 AI 입력 준비

### 학습 목표
근거·대안·추가 확인을 포함한 분석 보고를 작성한다.

### 배경 설명
label을 가리고 후보를 찾는다. 정상/공격 정답은 합성 시나리오일 뿐 현실의 자동 판정 규칙이 아니다.

### 코드
학생 수행 문제가 포함되어 있습니다.

In [ ]:
# *_answer는 학생 답안용 변수입니다. None은 아직 답을 작성하지 않았다는 표시입니다.
# night_admin_answer는 새벽 관리자 인증 성공 표, web_404_answer는 IP별 404 기록 수를 담는 자리입니다.
# label은 교수자 검증용이므로 필터 조건에 넣지 않습니다. report는 근거·정상 가능성·추가 확인을 적는 문자열입니다.
# 문제 7. 새벽 admin 인증 성공을 찾으세요.
night_admin_answer = None  # TODO: hour, username, result 조건
if night_admin_answer is not None:
    display(night_admin_answer)
# 문제 8. label을 사용하지 않고 404가 많은 IP를 찾으세요.
web_404_answer = None  # TODO: status_code 필터와 groupby
if web_404_answer is not None:
    display(web_404_answer.head())
# 문제 9. 관찰 / 정상 가능성 / 추가 확인을 각각 한 문장으로 쓰세요.
report = ''  # TODO: 3문장 보고서
print(report or '보고서를 작성하세요. 이상 징후와 공격 확정은 다릅니다.')

### 결과 확인 및 해석
새벽 admin 1건, 203.0.113.80의 404 112건. 종합 제출물: 표 1개, 그림 1개, 관찰·대안·추가 확인 3문장.

### 생각해보기
AI가 학습하기 전에 왜 전처리가 필요한가? label이나 event_id를 학습 입력에 넣으면 어떤 누수가 생길까?

## 참고 자료
- [Apache HTTP Server 로그 문서](https://httpd.apache.org/docs/2.4/logs.html)
- [pandas 시간 변환](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html)
- [pandas One-Hot Encoding](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html)
- [IPFIX 개념: RFC 7011](https://www.rfc-editor.org/rfc/rfc7011.html)
- [Colab FAQ](https://research.google.com/colaboratory/faq.html)

실습의 파서는 제공된 합성 형식만 지원합니다. 실제 장비의 모든 변형을 처리하는 범용 파서가 아닙니다.